In [ ]:
import os
import pandas as pd
import subprocess

def copy_to_hdfs(local_file, hdfs_path):
    container_file_path = f"/tmp/{os.path.basename(local_file)}"
    docker_cp_command = f"docker cp {local_file} {container_id}:{container_file_path}"
    subprocess.run(docker_cp_command, shell=True, check=True)
    hdfs_command = f'hdfs dfs -put "{container_file_path}" "{hdfs_path}"'
    docker_command = f'docker exec {container_id} bash -c "{hdfs_command}"'
    result = subprocess.run(docker_command, shell=True, check=True, capture_output=True, text=True)


container_id = "c3bbab323a1ca0c2eb556797e01b2e5631de808e2dbb05b19bf7427704f48d38"
local_output_dir = "C:/Users/User/Desktop/docker-hadoop/output"  
hdfs_result_path = "/result"  
df = pd.read_parquet('C:\\Users\\User\\Desktop\\docker-hadoop\\source.parquet')
df['ARR_TIME_ROUNDED'] = df['ARR_TIME'].round().astype(int)
df['CATEGORY'] = pd.cut(
    df['ARR_TIME_ROUNDED'],
    bins=[-float('inf'), 12, 18, float('inf')],
    labels=["morning", "afternoon", "evening"]
)
category_date_df = df.groupby("CATEGORY").agg(
    COUNT=("FL_DATE", "size"),
    SAMPLE_FL_DATE=("FL_DATE", "first")
).reset_index()
categories = category_date_df["CATEGORY"].unique()
for category in categories:
    category_df = df[df["CATEGORY"] == category]
    local_parquet_path = os.path.join(local_output_dir, f"{category}.parquet")
    category_df.to_parquet(local_parquet_path)
for file_name in os.listdir(local_output_dir):
    local_file = os.path.join(local_output_dir, file_name)
    if os.path.isfile(local_file):  
        copy_to_hdfs(local_file, hdfs_result_path)
print("Все работает")